In [43]:
## for chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
## convert text to doc
from langchain_community.document_loaders import TextLoader,DirectoryLoader
##utility
import numpy as np
import os
from typing import List
## to convert text into vector embeddings
from langchain_huggingface import embeddings
## to store vectors
import chromadb


os.makedirs("data",exist_ok=True)

text={
"data/py.txt":"""Python is one of the most popular programming languages in the world. 
It is widely used in web development, machine learning, artificial intelligence, 
automation, and data science because of its simple syntax and powerful libraries.""",

"data/emb.txt":"""Embeddings are numerical vector representations of text. 
They help machines understand semantic meaning by converting words, sentences, 
or documents into high-dimensional vectors that capture relationships between texts.""",


"data/cos.txt":"""Cosine similarity is a mathematical technique used to measure how similar two vectors are. 
Instead of comparing exact words, it compares the angle between vectors, making it useful 
for semantic search, recommendation systems, and Retrieval-Augmented Generation (RAG) applications.""",

"data/rag.txt":"""In a RAG pipeline, documents are converted into embeddings and stored in a vector database. 
When a user asks a question, the query is also converted into an embedding, and cosine similarity 
is used to retrieve the most relevant documents before generating the final answer."""
}

for file_path,content in text.items():
    with open(file_path,'w',encoding="utf-8") as f:
        f.write(content)


In [44]:
from langchain_chroma import Chroma
dirs = DirectoryLoader("data", glob="*.txt",loader_cls=TextLoader)
docs = dirs.load()
# print(docs)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20,separators=["\n\n","\n"," ",""])
chunks = text_splitter.split_documents(docs)
print(chunks)

# per_dir="./chroma_db"

chromaClient = chromadb.Client()
store = Chroma(
    collection_name="bhasrdsa"
)
# store = chromaClient.create_collection(name="bhasra")

## convert text to vector embeddings
# embed = embeddin+gs.HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



[Document(metadata={'source': 'data\\cos.txt'}, page_content='Cosine similarity is a mathematical technique used to measure how similar two vectors are.'), Document(metadata={'source': 'data\\cos.txt'}, page_content='Instead of comparing exact words, it compares the angle between vectors, making it useful'), Document(metadata={'source': 'data\\cos.txt'}, page_content='for semantic search, recommendation systems, and Retrieval-Augmented Generation (RAG) applications.'), Document(metadata={'source': 'data\\emb.txt'}, page_content='Embeddings are numerical vector representations of text.'), Document(metadata={'source': 'data\\emb.txt'}, page_content='They help machines understand semantic meaning by converting words, sentences,'), Document(metadata={'source': 'data\\emb.txt'}, page_content='or documents into high-dimensional vectors that capture relationships between texts.'), Document(metadata={'source': 'data\\py.txt'}, page_content='Python is one of the most popular programming languag

In [45]:
# store.add(
#     documents=[doc.page_content for i,doc in enumerate(chunks)],
#     ids=[str(i) for i, _ in enumerate(chunks)]
# )
store.add_documents(documents=chunks)
# store.get()
# print(store.count())
# print(store.get())

['f6692449-7ae5-41aa-9c4e-a706813ee89c',
 'a535c865-fd8e-4f2b-8e86-086fc3549985',
 'b8225dc2-de7f-4fc9-9de3-e8ea5f7836bb',
 '5ba7e26e-d8f2-42e3-94cd-0ea635989b13',
 '964d305e-c1a0-4cbc-af82-a3b56e66b451',
 '57fa1a0a-9097-4502-a2a3-a28274115804',
 '08b0286a-6101-40e1-ae90-0b632b01549f',
 '8d031169-832f-4869-ad02-99afaddeff76',
 'e3546c94-b0a3-47eb-a3ba-825a0884a3b2',
 'fd814466-1f80-49e2-a207-059e3f1d1feb',
 'e6360e95-6ea9-45e5-b7aa-2c85308c9835',
 'd56ceea6-9376-45c0-99f5-11dd58b157bf']

In [46]:
qu= "cosine"
# sim = store.query(query_texts=[query])
sim = store.similarity_search(query=qu)
# print(store.search(query))
# advsearch = store.get(where_document={"$contains":query})
print(sim)
# print(advsearch)

[Document(id='f6692449-7ae5-41aa-9c4e-a706813ee89c', metadata={'source': 'data\\cos.txt'}, page_content='Cosine similarity is a mathematical technique used to measure how similar two vectors are.'), Document(id='e6360e95-6ea9-45e5-b7aa-2c85308c9835', metadata={'source': 'data\\rag.txt'}, page_content='When a user asks a question, the query is also converted into an embedding, and cosine similarity'), Document(id='a535c865-fd8e-4f2b-8e86-086fc3549985', metadata={'source': 'data\\cos.txt'}, page_content='Instead of comparing exact words, it compares the angle between vectors, making it useful'), Document(id='8d031169-832f-4869-ad02-99afaddeff76', metadata={'source': 'data\\py.txt'}, page_content='It is widely used in web development, machine learning, artificial intelligence,')]


In [47]:
# from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="phi3"
)
test = llm.invoke("what is LLM")
test

AIMessage(content="LLM stands for Large Language Model, a type of artificial intelligence that has gained significant popularity and development in recent years. These models are designed to understand and generate human-like text by processing vast amounts of data (typically texts) to learn language patterns, grammar, context, idioms, and even the subtlet end rhymes or puns found within literary works like Shakespeare's plays.\n\nLarge Language Models such as GPT (Generative Pre-trained Transformer), BERT (Bidirectional Encoder Representations from Transformers), RoBERTa, etc., are capable of performing a wide variety of language tasks without needing task-specific training data or models. They can generate human-like text and often perform remarkably well on various natural language processing tasks such as translation, question answering, summarization among others because they understand the context in which words and sentences occur better than previous generation models due to th

In [48]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

ret = store.as_retriever(
    kwargs={"k":3}
)
print(ret)

tags=['Chroma'] vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000153B8417C10> search_kwargs={}


In [49]:
sys_prompt = """ you are an assistant for question-answering tasks. use the fallowing pieces of retrieved context to answer
the question. if you don't know the answer, just say that you don't know use three sentences maximum and 
keep the answers concise context:{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system",sys_prompt),
    ("human","{input}")
])

chain = create_stuff_documents_chain(llm,prompt)
# chain
rag_chain = create_retrieval_chain(ret,chain)
rag_chain.invoke({"input":"What is Python?"})


{'input': 'What is Python?',
 'context': [Document(id='08b0286a-6101-40e1-ae90-0b632b01549f', metadata={'source': 'data\\py.txt'}, page_content='Python is one of the most popular programming languages in the world.'),
  Document(id='e3546c94-b0a3-47eb-a3ba-825a0884a3b2', metadata={'source': 'data\\py.txt'}, page_content='automation, and data science because of its simple syntax and powerful libraries.'),
  Document(id='8d031169-832f-4869-ad02-99afaddeff76', metadata={'source': 'data\\py.txt'}, page_content='It is widely used in web development, machine learning, artificial intelligence,'),
  Document(id='964d305e-c1a0-4cbc-af82-a3b56e66b451', metadata={'source': 'data\\emb.txt'}, page_content='They help machines understand semantic meaning by converting words, sentences,')],
 'answer': "Python is a high-level programming language known for its readability and straightforward syntax. It's extensively utilized in various domains like web development, data analysis, automation tasks, mach